# RealPDE Eval on Kaggle (`local_eval.py`)

Editor-first flow: implement variants locally under `submissions/submission_vN/`, `git push`, then here just `git pull` and run. This notebook never writes `submission.py` — it only pulls the repo and calls `local_eval.py --submission submissions/<variant>`.

1. Pull repo → 2. resolve `--data` → 3. smoke-test each variant on `example_data/` → 4. (optional) score on real `test_real` + stage a baseline `model.pth` → 5. pack Codabench zips.

In [ ]:
# --- Clone or pull repo (public, no token) ---
REPO_DIR = '/kaggle/working/realpde'
REPO_URL = 'https://github.com/nthday-jpg/realpde.git'

!if [ -d {REPO_DIR} ]; then echo "Pulling {REPO_DIR}..."; cd {REPO_DIR} && git pull; else echo "Cloning {REPO_URL}..."; git clone {REPO_URL} {REPO_DIR}; fi

In [ ]:
%cd {REPO_DIR}
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import torch
print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# --- Resolve --data (example_data fallback, real test_real if attached) ---
from pathlib import Path

REPO = Path(REPO_DIR)
example_data = REPO / 'example_data'
DATA_DIR = example_data  # default: bundled synthetic smoke test
for cand in [Path('/kaggle/input/realpde'), Path('/kaggle/input/realpde/test_real').parent]:
    if (cand / 'test_real').is_dir() and (cand / 'mean_std_real.pt').exists():
        DATA_DIR = cand
        break
print(f'DATA_DIR -> {DATA_DIR}')
print(f'  example_data present: {(example_data / "test_real").is_dir()}')

# --- List submission variants ---
variants = sorted(p.name for p in (REPO / 'submissions').glob('submission_*') if p.is_dir())
print(f'variants: {variants}')

In [ ]:
# --- Smoke test every variant on example_data (CPU, always works) ---
import subprocess, sys
from pathlib import Path

REPO = Path(REPO_DIR)
for v in sorted(p.name for p in (REPO / 'submissions').glob('submission_*') if p.is_dir()):
    print(f'\n===== {v} (example_data) =====')
    r = subprocess.run([sys.executable, 'local_eval.py', '--submission', f'submissions/{v}',
                        '--data', './example_data'], cwd=REPO)
    print(f'[{v}] exit code: {r.returncode}')

In [ ]:
# --- (Optional) baseline checkpoint: stage model.pth into each variant, re-run ---
# Finds a CNO checkpoint under /kaggle/input/realpde (or training output in
# /kaggle/working/checkpoints), copies it as model.pth (gitignored, never
# committed), and re-runs local_eval so you see the real-weights numbers.
import shutil, subprocess, sys
from pathlib import Path

REPO = Path(REPO_DIR)
search_roots = [Path('/kaggle/input/realpde'), Path('/kaggle/working/checkpoints')]
ckpts = []
for root in search_roots:
    if root.exists():
        ckpts += list(root.rglob('*.pth'))
cno = sorted([p for p in ckpts if 'cno' in p.name.lower()],
             key=lambda p: (0 if 'sim_real' in p.name.lower() else 1, p.name))
ckpt = cno[0] if cno else (sorted(ckpts)[0] if ckpts else None)
print(f'checkpoint: {ckpt}')
if ckpt is None:
    print('[skip] no checkpoint found — variants run with TinyForecaster fallback')
else:
    for v in sorted(p.name for p in (REPO / 'submissions').glob('submission_*') if p.is_dir()):
        dst = REPO / 'submissions' / v / 'model.pth'
        shutil.copy(ckpt, dst)
        print(f'[stage] {ckpt.name} -> submissions/{v}/model.pth '
              f'({dst.stat().st_size / 1e6:.1f} MB)')
    data_arg = str(DATA_DIR)
    for v in sorted(p.name for p in (REPO / 'submissions').glob('submission_*') if p.is_dir()):
        print(f'\n===== {v} ({data_arg}, with model.pth) =====')
        r = subprocess.run([sys.executable, 'local_eval.py', '--submission', f'submissions/{v}',
                            '--data', data_arg], cwd=REPO)
        print(f'[{v}] exit code: {r.returncode}')

In [ ]:
# --- Pack Codabench zips (submission.py at root, shared files injected) ---
import subprocess, sys
from pathlib import Path

REPO = Path(REPO_DIR)
for v in sorted(p.name for p in (REPO / 'submissions').glob('submission_*') if p.is_dir()):
    print(f'\n===== packing {v} =====')
    r = subprocess.run([sys.executable, 'scripts/make_submission_zip.py', v], cwd=REPO)
    print(f'[{v}] pack exit code: {r.returncode}')
print('\nZips:')
!ls -la dist/ 2>/dev/null || echo '(no dist/ yet — pack a variant first)'